# Reasoning Models and Test-Time Compute

The full code of [Reasoning Models and Test-Time Compute](https://residual-stream.lvtan71.workers.dev/posts/reasoning-and-test-time-compute/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/reasoning-and-test-time-compute) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/reasoning-and-test-time-compute.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 34 min end to end on the RTX 3090 machine that saved the outputs below.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB). Sampled answers vary by machine; hard_stats.py reads the post's saved tts_hard.out, not a new run's output.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 34 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.18.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "reasoning-and-test-time-compute"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/reasoning-and-test-time-compute


## `cost.py`

What thinking tokens cost on one H100 for Qwen2.5-32B and DeepSeek-R1: KV cache size and bandwidth-bound decode time at 1,000, 10,000 and 50,000 tokens.

In [2]:
!{sys.executable} cost.py

              KV per token   thinking tokens  KV cache  decode time  avg tokens/s
Qwen2.5-32B      256.0 KiB             1,000    0.24 GiB       19.4 s            51
Qwen2.5-32B      256.0 KiB            10,000    2.44 GiB      197.9 s            51
Qwen2.5-32B      256.0 KiB            50,000   12.21 GiB     1068.0 s            47
DeepSeek-R1       68.6 KiB             1,000    0.07 GiB       11.1 s            90
DeepSeek-R1       68.6 KiB            10,000    0.65 GiB      111.5 s            90
DeepSeek-R1       68.6 KiB            50,000    3.27 GiB      578.5 s            86


The post printed:

```text
              KV per token   thinking tokens  KV cache  decode time  avg tokens/s
Qwen2.5-32B      256.0 KiB             1,000    0.24 GiB       19.4 s            51
Qwen2.5-32B      256.0 KiB            10,000    2.44 GiB      197.9 s            51
Qwen2.5-32B      256.0 KiB            50,000   12.21 GiB     1068.0 s            47
DeepSeek-R1       68.6 KiB             1,000    0.07 GiB       11.1 s            90
DeepSeek-R1       68.6 KiB            10,000    0.65 GiB      111.5 s            90
DeepSeek-R1       68.6 KiB            50,000    3.27 GiB      578.5 s            86
```

## `tts_seeds.py`

Runs `tts.py` with seeds 0 and 1, as the post did.

In [3]:
!{sys.executable} tts_seeds.py

seed 0: 100 problems, 1135 s
  greedy, direct: 1% right, 3 tokens per answer
  greedy, chain : 100% right, 217 tokens per answer
  sampled chain, one sample: 97% right, 175 tokens per answer
  majority of  1: 96% right (  175 tokens per question); some sample right: 96%
  majority of  2: 96% right (  350 tokens per question); some sample right: 100%
  majority of  4: 100% right (  700 tokens per question); some sample right: 100%
  majority of  8: 100% right ( 1399 tokens per question); some sample right: 100%
  majority of 16: 100% right ( 2799 tokens per question); some sample right: 100%


seed 1: 100 problems, 648 s
  sampled chain, one sample: 97% right, 176 tokens per answer
  majority of  1: 96% right (  176 tokens per question); some sample right: 96%
  majority of  2: 97% right (  352 tokens per question); some sample right: 100%
  majority of  4: 100% right (  704 tokens per question); some sample right: 100%
  majority of  8: 100% right ( 1409 tokens per question); some sample right: 100%
  majority of 16: 100% right ( 2817 tokens per question); some sample right: 100%


The post printed:

```text
seed 0: 100 problems, 2570 s
  greedy, direct: 1% right, 3 tokens per answer
  greedy, chain : 100% right, 215 tokens per answer
  sampled chain, one sample: 97% right, 178 tokens per answer
  majority of  1: 97% right (  178 tokens per question); some sample right: 97%
  majority of  2: 98% right (  357 tokens per question); some sample right: 100%
  majority of  4: 100% right (  713 tokens per question); some sample right: 100%
  majority of  8: 100% right ( 1426 tokens per question); some sample right: 100%
  majority of 16: 100% right ( 2853 tokens per question); some sample right: 100%
seed 1: 100 problems, 1828 s
  sampled chain, one sample: 98% right, 175 tokens per answer
  majority of  1: 99% right (  175 tokens per question); some sample right: 99%
  majority of  2: 99% right (  351 tokens per question); some sample right: 100%
  majority of  4: 100% right (  702 tokens per question); some sample right: 100%
  majority of  8: 100% right ( 1404 tokens per question); some sample right: 100%
  majority of 16: 100% right ( 2808 tokens per question); some sample right: 100%
```

## `tts_hard.py`

40 problems of eight dependent operations, 16 sampled chains each: majority of k, some sample right, and one line per problem.

In [4]:
!{sys.executable} tts_hard.py

[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


problem 1: 15/16 samples right


problem 2: 16/16 samples right


problem 3: 15/16 samples right


problem 4: 15/16 samples right


...

problem 36: 14/16 samples right


problem 37: 16/16 samples right


problem 38: 16/16 samples right


problem 39: 15/16 samples right


problem 40: 15/16 samples right

40 problems, eight operations each; 201 tokens per sample
  majority of  1:  85% right (   201 tokens per question); some sample right:  85%
  majority of  2:  85% right (   402 tokens per question); some sample right:  90%
  majority of  4:  92% right (   804 tokens per question); some sample right:  92%
  majority of  8:  92% right (  1608 tokens per question); some sample right:  98%
  majority of 16:  92% right (  3217 tokens per question); some sample right:  98%
problems by share of samples right: none 1, under half 3, half or more 22, all 14


The post printed:

```text
problem 1: 15/16 samples right
problem 2: 14/16 samples right
problem 3: 14/16 samples right
problem 4: 15/16 samples right
problem 5: 13/16 samples right
problem 6: 15/16 samples right
problem 7: 16/16 samples right
problem 8: 0/16 samples right
problem 9: 15/16 samples right
problem 10: 15/16 samples right
problem 11: 14/16 samples right
problem 12: 15/16 samples right
problem 13: 15/16 samples right
problem 14: 16/16 samples right
problem 15: 16/16 samples right
problem 16: 15/16 samples right
problem 17: 15/16 samples right
problem 18: 12/16 samples right
problem 19: 15/16 samples right
problem 20: 15/16 samples right
problem 21: 1/16 samples right
problem 22: 16/16 samples right
problem 23: 16/16 samples right
problem 24: 14/16 samples right
problem 25: 4/16 samples right
problem 26: 15/16 samples right
problem 27: 15/16 samples right
problem 28: 9/16 samples right
problem 29: 13/16 samples right
problem 30: 16/16 samples right
problem 31: 15/16 samples right
problem 32: 16/16 samples right
problem 33: 16/16 samples right
problem 34: 15/16 samples right
problem 35: 14/16 samples right
problem 36: 15/16 samples right
problem 37: 16/16 samples right
problem 38: 16/16 samples right
problem 39: 16/16 samples right
problem 40: 16/16 samples right

40 problems, eight operations each; 202 tokens per sample
  majority of  1:  82% right (   202 tokens per question); some sample right:  82%
  majority of  2:  88% right (   403 tokens per question); some sample right:  90%
  majority of  4:  92% right (   807 tokens per question); some sample right:  95%
  majority of  8:  92% right (  1614 tokens per question); some sample right:  95%
  majority of 16:  92% right (  3227 tokens per question); some sample right:  98%
problems by share of samples right: none 1, under half 2, half or more 25, all 12
```

## `hard_stats.py`

Average one-sample accuracy and unbiased pass@k from the per-problem counts in `tts_hard.out`.

**Note:** Reads tts_hard.out, the post's saved log of tts_hard.py.

In [5]:
!{sys.executable} hard_stats.py

40 problems; one sample right on average: 86.6%
  pass@1  (some of  1 samples right): 86.6%
  pass@2  (some of  2 samples right): 93.1%
  pass@4  (some of  4 samples right): 94.9%
  pass@8  (some of  8 samples right): 96.2%
  pass@16 (some of 16 samples right): 97.5%


The post printed:

```text
40 problems; one sample right on average: 86.6%
  pass@1  (some of  1 samples right): 86.6%
  pass@2  (some of  2 samples right): 93.1%
  pass@4  (some of  4 samples right): 94.9%
  pass@8  (some of  8 samples right): 96.2%
  pass@16 (some of 16 samples right): 97.5%
```